In [ ]:
#impot required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import shap
import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 42


In [ ]:
#data preprocessing  
# Load dataset
df = pd.read_csv('your_dataset.csv')  # <-- change file name

# Separate features and target
X = df.drop('target', axis=1)  # <-- change target column name
y = df['target']


# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)


In [ ]:
# Model training 
models = {
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        eval_metric='logloss'
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=RANDOM_STATE,
        verbose=0
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=RANDOM_STATE
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)


In [ ]:
# Test Set Evaluation-  Matrixrics, Confusion Matrix, Classification Report, ROC Curve
def evaluate_model(name, model):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"\n{name} Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print("Confusion Matrix:")
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix Heatmap
    plt.figure()
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks([0.5,1.5], ['Non-Class (0)', 'Class (1)'])
    plt.yticks([0.5,1.5], ['Non-Class (0)', 'Class (1)'])
    plt.show()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0,1],[0,1],'--')
    plt.title(f"ROC Curve - {name}")
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.show()

for name, model in models.items():
    evaluate_model(name, model)


In [ ]:
# Cross-Validation 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = ['accuracy','precision','recall','f1','roc_auc']

for name, model in models.items():
    scores = cross_validate(model, X_scaled, y, cv=cv, scoring=scoring)
    print(f"\n{name} Cross-Validation Results:")
    for metric in scoring:
        mean = scores[f'test_{metric}'].mean()
        std = scores[f'test_{metric}'].std()
        print(f"{metric}: {mean:.4f} ± {std:.4f}")


In [ ]:
#SHAP Interpretation
X_test_df = pd.DataFrame(X_test, columns=X.columns)

for name, model in models.items():
    print(f"\nSHAP for {name}")
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_df)

    # Summary plot
    shap.summary_plot(shap_values, X_test_df)

    # Feature importance bar plot
    shap.summary_plot(shap_values, X_test_df, plot_type='bar')


### Report?
